In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    "lppls_rolling_results.csv",
    parse_dates=["win_start", "win_end"]
)
df.head()

,win_start,win_end,n,ok,tc,m,w,A,B,C,c1,c2,O,D,ssr,r2,cond_m,cond_w,cond_B,cond_tc
0,2015-01-01,2015-01-30,30,True,7.355947e+05,0.966525,6.720100,5.798027,-1.953252e-02,-9.690943e-03,-7.015534e-03,-6.685556e-03,-2.195625,0.289888,0.063431,0.894214,True,True,True,False
1,2015-01-01,2015-01-31,31,True,7.356358e+05,0.692946,6.055410,5.450283,2.417122e-03,-1.917749e-02,-1.374716e-02,1.337131e-02,1.625319,0.014423,0.117710,0.808814,True,True,False,True
2,2015-01-01,2015-02-01,32,True,8.063716e+05,0.767180,5.289823,0.063901,5.781582e+01,5.781480e+01,4.708358e+01,-3.355126e+01,0.000369,0.145032,0.299048,0.519356,True,False,False,False
3,2015-01-01,2015-02-02,33,True,8.216148e+05,0.700389,5.650351,0.129405,1.311957e+02,-1.311938e+02,-2.646339e+01,-1.284971e+02,0.000335,0.123957,0.301119,0.516747,True,False,False,False
4,2015-01-01,2015-02-03,34,True,2.293064e+06,5.738598,126.985986,0.087918,6.442018e-31,-6.441863e-31,-3.039593e-31,-5.679654e-31,0.000428,0.045192,0.309646,0.507623,False,False,False,False


In [4]:
data_union = [
    ["UNION01", "2016-06-08", "2016-06-18"],
    ["UNION02", "2016-12-30", "2017-01-15"],
    ["UNION03", "2017-02-22", "2017-03-07"],
    ["UNION04", "2017-04-27", "2017-07-05"],
    ["UNION05", "2017-07-15", "2018-01-09"],
    ["UNION06", "2019-06-17", "2019-07-13"],
    ["UNION07", "2019-08-09", "2019-08-10"],
    ["UNION08", "2019-12-06", "2020-01-05"],
    ["UNION09", "2020-02-07", "2020-02-17"],
    ["UNION10", "2020-03-20", "2020-09-21"],
    ["UNION11", "2020-11-06", "2021-05-20"],
    ["UNION12", "2021-08-08", "2021-09-19"],
    ["UNION13", "2021-10-05", "2021-12-02"],
    ["UNION14", "2024-03-04", "2024-03-25"],
]

bubbles = pd.DataFrame(
    data_union,
    columns=["label_id", "start_date", "end_date"]
)
bubbles["start_date"] = pd.to_datetime(bubbles["start_date"])
bubbles["end_date"] = pd.to_datetime(bubbles["end_date"])

bubbles.head()

,label_id,start_date,end_date
0,UNION01,2016-06-08,2016-06-18
1,UNION02,2016-12-30,2017-01-15
2,UNION03,2017-02-22,2017-03-07
3,UNION04,2017-04-27,2017-07-05
4,UNION05,2017-07-15,2018-01-09


In [5]:
# 3. Utworzenie nowych kolumn

df["literature_bubble"] = 0
df["bubble_id"] = pd.NA
df["bubble_overlap_days"] = 0

window_days = (df["win_end"] - df["win_start"]).dt.days + 1


# 4. Etykietowanie okien LPPLS

for _, bubble in bubbles.iterrows():

    start = bubble["start_date"]
    end = bubble["end_date"]
    label = bubble["label_id"]

    # Czy dzień końca okna przypada w bańce?
    inside = df["win_end"].between(start, end)

    df.loc[inside, "literature_bubble"] = 1
    df.loc[inside, "bubble_id"] = label

    # Ile dni okna pokrywa się z bańką?
    overlap_start = df["win_start"].clip(lower=start)
    overlap_end = df["win_end"].clip(upper=end)

    overlap_days = (
        (overlap_end - overlap_start)
        .dt.days
        .add(1)
        .clip(lower=0)
    )

    df["bubble_overlap_days"] += overlap_days


# 5. Procent pokrycia okna przez bańki

df["bubble_overlap_ratio"] = (
    df["bubble_overlap_days"] / window_days
)


In [6]:
output_file = "lppls_rolling_results_z_etykietami_baniek.csv"

df.to_csv(output_file, index=False)


In [7]:
print("Liczba wszystkich okien:", len(df))

print("\nRozkład etykiet:")
print(df["literature_bubble"].value_counts())

print("\nLiczba okien w poszczególnych bańkach:")
print(df["bubble_id"].value_counts().sort_index())

print("\nZapisano:", output_file)


Liczba wszystkich okien: 107240

Rozkład etykiet:
literature_bubble
0    81026
1    26214
Name: count, dtype: int64

Liczba okien w poszczególnych bańkach:
bubble_id
UNION01     333
UNION02     514
UNION03     423
UNION04    2114
UNION05    5406
UNION06     816
UNION07      60
UNION08     936
UNION09     332
UNION10    5617
UNION11    5919
UNION12    1298
UNION13    1782
UNION14     664
Name: count, dtype: int64

Podgląd danych:
   win_start    win_end  literature_bubble bubble_id  bubble_overlap_days  \
0 2015-01-01 2015-01-30                  0      <NA>                    0   
1 2015-01-01 2015-01-31                  0      <NA>                    0   
2 2015-01-01 2015-02-01                  0      <NA>                    0   
3 2015-01-01 2015-02-02                  0      <NA>                    0   
4 2015-01-01 2015-02-03                  0      <NA>                    0   
5 2015-01-01 2015-02-04                  0      <NA>                    0   
6 2015-01-01 2015-02-05     